# One real clock: the JILA Sr budget slice

Every validation notebook in this project checks one term against one
published number. This notebook points the same machinery at a single
real platform, the JILA Sr-87 lattice clock, and computes the slice of
its published systematic evaluation this tool covers, from published
inputs, beside the lab's own numbers. Two papers supply everything:
Aeppli et al. (PRL 133, 023401 (2024)) published the platform's current
evaluation, including the blackbody-radiation row and the operating
temperature behind it, and Bothwell et al. (Nature 602, 420 (2022))
measured the gravitational redshift across a millimetre-scale sample on
the same platform, with a systematic budget of its own. The sections
below compose the two validation cases notebooks 07 and 09 walk through
individually, add the one published row that is in-scope physics
without a public input to run on, and close with the rows this tool
does not cover, listed from the committed report's own notes.

**Status: pre-beta research code.**

In [1]:
import copy
import json
import re
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "benchmarks"))

import run_bbr_jila_arithmetic_reproduction as jila_bbr
import run_bothwell_redshift as bothwell

from cliffordclock.ensemble.species import get_species
from cliffordclock.integrator.omega import bbr_pivot_perturbation
from cliffordclock.pipeline import PipelineConfig, run_pipeline_full

## 1. The BBR row: E32 at their measured temperature

A room-temperature vacuum chamber bathes the atoms in thermal
radiation, and E32 (`docs/CONVENTIONS.md` section 13) writes the
resulting pivot term as a `(T/300K)^4` static piece plus a small
per-species dynamic polynomial: the same second-order Stark mechanism
as a static field, averaged over the thermal photon bath. Its two
inputs are both published: Aeppli et al.'s in-vacuum
resistance-thermometer temperature, T = 293.282(4) K, and the Sr-87
registry's BBR coefficients, each printed with its citation below. The
committed case object (`benchmarks/run_bbr_jila_arithmetic_reproduction.py`,
walked in full in notebook 07) evaluates the real engine functions at
that temperature and compares the result against the paper's own
Table I BBR row.

In [2]:
bbr_case = jila_bbr.run_jila_bbr_arithmetic_reproduction_case()

print(
    f"temperature (Aeppli et al.):  {bbr_case.temperature_k:.3f} "
    f"+/- {bbr_case.temperature_uncertainty_k:.3f} K"
)
print(f"  citation: {bbr_case.temperature_citation}")
print()
print("registry coefficients:")
print(f"  {bbr_case.registry_coefficient_citation}")
print()
print(f"predicted (P-1)_BBR:      {bbr_case.predicted_shift_nominal:+.6e}")
print(
    f"predicted combined band:  [{bbr_case.predicted_combined_band_lo:+.6e}, "
    f"{bbr_case.predicted_combined_band_hi:+.6e}]"
)
print(
    f"published band (Table I): [{bbr_case.published_shift_lo:+.6e}, "
    f"{bbr_case.published_shift_hi:+.6e}]  nominal {bbr_case.published_shift_nominal:+.6e}"
)
print(f"  citation: {bbr_case.published_shift_citation}")
print()
print(f"residual (predicted - published): {bbr_case.residual_fractional:+.3e}")
print(f"kpi_verdict: {bbr_case.kpi_verdict}")
print()
print(f"case_class: {bbr_case.case_class!r}")
print(f"case_label: {bbr_case.case_label}")

temperature (Aeppli et al.):  293.282 +/- 0.004 K
  citation: Aeppli, Kim, Warfield, Safronova, Ye, arXiv:2403.10664v2 (PRL 133, 023401 (2024)), main-text in-vacuum RTD temperature statement: "T = 20.132(4) degC = 293.282(4) K"; cross-checked against the primary text in this project's internal review sweep (2026-08-11) -- see benchmarks/SOURCES.md section 1 for the already-logged arXiv:2403.10664 fetch/checksum this value extends.

registry coefficients:
  cliffordclock.ensemble.species.SR87.bbr_coefficients: static nu_stat_300k_hz=-2.13023(6) Hz (T. Middelmann, S. Falke, C. Lisdat, U. Sterr, Phys. Rev. Lett. 109, 263004 (2012)); dynamic dyn_coeffs_hz={6: -0.13216, 8: -0.01231, 10: -0.00858} Hz, the PTB-2025 rescaled polynomial (arXiv:2507.14030) -- Lisdat et al. PR Research 3, L042036 (2021)'s fit SHAPE rescaled to Aeppli et al. arXiv:2403.10664 (2024)'s own -153.06(33) mHz dynamic-term ANCHOR at 300 K. See the project's theory sign-off record (G7) B2 for the shape-vs-anchor reasoning

The predicted band lands inside the published band with a residual two
orders of magnitude below either band's half width, and the class
label printed above is the binding one: an arithmetic reproduction,
expected almost by construction, because the registry's dynamic
coefficients are anchored to this same group's own published
dynamic-term value, so the case demonstrates the engine's arithmetic
and provenance chain end to end and nothing stronger. That is this
section's one caveat, and notebook 07 unpacks it across the E32
formula's full temperature validity window.

## 2. The gravitational-redshift row: E36 across their actual sample

General Relativity's leading laboratory-scale effect is linear in
height: a clock's fractional frequency shift relative to a reference
height is `g*(h - h_ref)/c^2`, which `docs/CONVENTIONS.md` E36
(section 15) composes additively into the same scalar pivot the BBR
term occupies. Bothwell et al. measured that gradient across a
millimetre of Sr-87, and the pipeline's `ensemble.regime:
lattice_extended` rebuilds their sample from published statements:
sites every 406.5 nm (half the 813 nm magic wavelength), a Gaussian
occupation envelope whose width follows from their published imaging
window, and Boulder's USGS-surveyed local g = 9.796 m/s^2 in place of
the standard-gravity default. The cell below runs the identical
machinery live at a reduced ~200-site window, exactly as notebook 09
does, and the cell after it loads the committed benchmark, whose
full-size-grid values are the ones actually compared against the
paper; the field-free gravity term is exactly linear in height, so the
fitted slope is site-count independent and the two must agree.

In [3]:
N_SITES_DEMO = 201  # reduced live window; the committed benchmark runs the full grid

print(
    f"site spacing:     {bothwell.SITE_SPACING_M * 1e9:.1f} nm "
    f"(= {bothwell.MAGIC_WAVELENGTH_M * 1e9:.0f} nm magic wavelength / 2, INFERRED)"
)
print(
    f"envelope sigma:   {bothwell.ENVELOPE_SIGMA_M * 1e6:.1f} um "
    f"(imaging pixel size + analysis window, INFERRED)"
)
print(f"surveyed local g: {bothwell.BOTHWELL_SURVEYED_G_M_S2} m/s^2 (Boulder, CO)")

BOTHWELL_DEMO_CONFIG_DICT = {
    "species": "Sr87",
    "trap": {"omega_xyz": [2.0e5, 2.0e5, 2.0e5], "center": [0.0, 0.0, 0.0]},
    "field": {"synthetic": {"kind": "uniform", "params": {"e0": [0.0, 0.0, 0.0]}}},
    "coupling": {"type": "stark_dc"},
    "ensemble": {
        "regime": "lattice_extended",
        "temperature_uK": 1.0,
        "motional_n": [0, 0, 0],
        "n_quad": 1,
        "n_sites": N_SITES_DEMO,
        "site_spacing_m": bothwell.SITE_SPACING_M,
        "site_axis": [0.0, 0.0, 1.0],
        "site_envelope": "gaussian",
        "site_envelope_sigma_m": bothwell.ENVELOPE_SIGMA_M,
    },
    "integration": {"mode": "fast_path", "time_s": 1.0},
    "environment": {
        "gravity": {
            "g_m_s2": bothwell.BOTHWELL_SURVEYED_G_M_S2,
            "up_axis": [0.0, 0.0, 1.0],
            "reference_height_m": 0.0,
        }
    },
}
demo_config = PipelineConfig.from_dict(BOTHWELL_DEMO_CONFIG_DICT)
demo_result = run_pipeline_full(demo_config)
demo_site_map = demo_result.site_map
assert demo_site_map is not None
demo_slope_per_mm = -demo_site_map.slope_per_m / 1000.0  # Bothwell's coordinate convention

print()
print(f"sites in this live run:      {len(demo_site_map.sites)}")
print(f"slope (engine convention):   {demo_site_map.slope_per_m:+.6e} /m")
print(f"slope (Bothwell convention): {demo_slope_per_mm:+.4e} /mm")

site spacing:     406.5 nm (= 813 nm magic wavelength / 2, INFERRED)
envelope sigma:   402.7 um (imaging pixel size + analysis window, INFERRED)
surveyed local g: 9.796 m/s^2 (Boulder, CO)



sites in this live run:      201
slope (engine convention):   +1.089952e-16 /m
slope (Bothwell convention): -1.0900e-19 /mm


In [4]:
full_case_path = Path("..") / "benchmarks" / "results" / "bothwell_redshift.json"
full_report = json.loads(full_case_path.read_text(encoding="utf-8"))
grav_case = full_report["bothwell_2022_nature_602_420_redshift_case"]

print(f"full grid site count:        {grav_case['n_sites']}")
print(f"  g citation: {grav_case['g_citation']}")
print()
print(f"predicted slope (full grid): {grav_case['predicted_slope_per_mm']:+.4e} /mm")
print(f"predicted slope (live run):  {demo_slope_per_mm:+.4e} /mm")
print()
for method_key, label in (
    ("a", "method A (14-dataset campaign)"),
    ("b", "method B (synchronous two-region)"),
):
    measured = grav_case[f"measured_slope_method_{method_key}"]
    print(f"{label}:")
    print(
        f"  measured {measured['nominal']:+.2e} [{measured['lo']:+.2e}, {measured['hi']:+.2e}] /mm"
    )
    print(
        f"  sigma distance: {grav_case[f'sigma_distance_method_{method_key}']:.2f}   "
        f"kpi_verdict: {grav_case[f'kpi_verdict_method_{method_key}']}"
    )
print()
print(f"case_class: {grav_case['case_class']!r}")
print(f"case_label: {grav_case['case_label']}")

full grid site count:        5945
  g citation: van Westrum, D., NOAA Technical Memorandum NOS NGS-77 (2019), as cited in Bothwell et al., Nature 602, 420 (2022) / arXiv:2109.12238 Methods 'Known Redshift': USGS-surveyed local g = 9.796 m/s^2 at the JILA Boulder, CO site.

predicted slope (full grid): -1.0900e-19 /mm
predicted slope (live run):  -1.0900e-19 /mm

method A (14-dataset campaign):
  measured -9.80e-20 [-1.21e-19, -7.50e-20] /mm
  sigma distance: 0.48   kpi_verdict: MET
method B (synchronous two-region):
  measured -1.28e-19 [-1.55e-19, -1.01e-19] /mm
  sigma distance: 0.70   kpi_verdict: MET

case_class: 'reproducibility'
case_label: reproducibility, with the INVERTED-NPL caveat: the g/c^2 arithmetic is textbook and the authors computed it themselves trivially (unlike NPL's differential-polarizability reconstruction); what this case validates is the extended-sample MACHINERY (per-site geometry, Gaussian-envelope weighting, map assembly) producing the right measured-map slo

Both of the paper's independently analyzed corrected measurements
bracket the prediction, at 0.48 sigma from the fourteen-dataset
campaign and 0.70 sigma from the synchronous two-region comparison,
with zero adjustable inputs anywhere in the chain. The label printed
above carries this section's one caveat in the record's own wording:
the g/c^2 arithmetic is textbook and Bothwell computed it themselves,
so what this row validates is the extended-sample machinery, the
per-site geometry, envelope weighting, and map fit, producing their
measured-map slope end to end.

## 3. The DC-Stark gradient row: in scope, awaiting one unpublished input

Bothwell's Table 1 budget also carries a DC-Stark gradient row, and the
committed case ships a context note holding that row's published value,
printed verbatim below. The physics is E14b, the same coupling every
field-driven notebook in this project runs on real chamber maps, so the
engine could compute this row today from exactly one input the paper
does not publish: the residual electric field characterization inside
their chamber. Without that field map there is no predicted number to
place beside their published one, and this notebook does not
manufacture a comparison; the row stays context. A lab sharing that
field characterization ahead of, or independently of, its own shift
measurement is precisely the partner arrangement `docs/roadmap.md` asks
for, and this row is the concrete first candidate for it.

In [5]:
print(grav_case["dc_stark_context_note"])

DC_STARK_ROW_VALUE = re.search(
    r"[+-]\d[\d.]*\(\d[\d.]*\)e-20/mm", grav_case["dc_stark_context_note"]
).group(0)
print()
print(f"published row value, extracted from the note above: {DC_STARK_ROW_VALUE}")

Bothwell's own DC-Stark gradient row, +0.3(0.2)e-20/mm (their Table 1 budget-level corrections), is in-scope physics for this engine (CONVENTIONS.md E14b) but is a separate systematic this case does not model (the comparison target already has it corrected out, see isolation_note); it enters this report as a narrative cross-reference only.

published row value, extracted from the note above: +0.3(0.2)e-20/mm


## 4. One composed run: both covered terms in a single report

The two rows above ran as isolated cases, and a real evaluation carries
every active term in one budget, so this section reruns the
reduced-site Bothwell geometry with a single addition:
`environment.radiation_temperature_K` set to Aeppli et al.'s measured
temperature and its published uncertainty, the field still exactly zero
to match the committed benchmark's isolation logic. Two printed checks
make the composition concrete: the composed mean shift minus the
gravity-only mean shift must land on the direct E32 call to
floating-point roundoff, and the fitted per-site slope must be
untouched, because a spatially uniform BBR term moves every site
identically while gravity alone sets the gradient. The report's
`uncertainty_notes`, printed in full, then state in the tool's own
words what is in the composed number and what is excluded from it.

In [6]:
composed_config_dict = copy.deepcopy(BOTHWELL_DEMO_CONFIG_DICT)
composed_config_dict["environment"]["radiation_temperature_K"] = bbr_case.temperature_k
composed_config_dict["environment"]["radiation_temperature_uncertainty_K"] = (
    bbr_case.temperature_uncertainty_k
)
composed_config = PipelineConfig.from_dict(composed_config_dict)
composed_result = run_pipeline_full(composed_config)
composed_site_map = composed_result.site_map
assert composed_site_map is not None

sr87 = get_species("Sr87")
bbr_direct = bbr_pivot_perturbation(bbr_case.temperature_k, sr87)
composed_mean = composed_result.report.mean_fractional_shift
gravity_mean = demo_result.report.mean_fractional_shift

print(f"gravity-only mean_fractional_shift:  {gravity_mean:+.6e}")
print(f"composed mean_fractional_shift:      {composed_mean:+.6e}")
print(f"composed - gravity-only (BBR term):  {composed_mean - gravity_mean:+.6e}")
print(f"bbr_pivot_perturbation(T) direct:    {bbr_direct:+.6e}")
bbr_agreement = abs((composed_mean - gravity_mean) - bbr_direct)
print(f"agreement (abs diff):                {bbr_agreement:.3e}")
print()
print(f"composed slope:     {-composed_site_map.slope_per_m / 1000.0:+.4e} /mm")
print(f"gravity-only slope: {demo_slope_per_mm:+.4e} /mm")
print(
    f"slope abs diff:     {abs(composed_site_map.slope_per_m - demo_site_map.slope_per_m):.3e} /m"
)
print()
print("report.uncertainty_notes:")
print(composed_result.report.uncertainty_notes)

gravity-only mean_fractional_shift:  +0.000000e+00
composed mean_fractional_shift:      -4.841743e-15
composed - gravity-only (BBR term):  -4.841743e-15
bbr_pivot_perturbation(T) direct:    -4.841743e-15
agreement (abs diff):                7.889e-31

composed slope:     -1.0900e-19 /mm
gravity-only slope: -1.0900e-19 /mm
slope abs diff:     7.674e-29 /m

report.uncertainty_notes:
integration.mode=fast_path T=1.0s (E29, exact; ensemble.regime=lattice_extended, WP22, n_sites=201, site_spacing_m=4.065e-07, site_envelope='gaussian') coupling=stark_dc (E14b): k_S=-3.077789630705917e-06 Hz.m^-2.V^-2, nu_0=429228004229873.4 Hz, source=species registry entry for 'Sr87' (Middelmann et al., Phys. Rev. Lett. 109, 263004 (2012)) fast_path (E29) reports the Stark/field shift only: static v=0 quadrature nodes carry no motional second-order Doppler contribution (CONVENTIONS.md E29 scope); that term is a separate, real clock systematic not included in this run's mean_fractional_shift. BBR (CONVENTION

Read the note's exclusion sentences closely, because they draw the
boundary of the covered slice. The fast-path scope sentence excludes
the motional second-order Doppler contribution, the BBR sentence models
out the M1/E2 multipole contributions and states their magnitude, and
the dispersion-labeling sentence separates the deterministic height
gradient from stochastic spread so a reader cannot mistake one for the
other. Nothing in the note claims lattice-light, density, or Zeeman
coverage, because those terms are computed nowhere in this release, and
the closing section lists them from the committed record itself.

## 5. The covered slice, summarized

Every number in the table below is pulled from the live case objects
and the committed benchmark JSON already printed above. Each row keeps
one unit convention across all of its columns, the published value
carries its stated uncertainty, and the two added columns give the
difference (this tool minus published) and that difference in units of
the published uncertainty, so agreement is a number, never an
impression. Dividing by the published uncertainty alone is the
conservative convention: it ignores this tool's own prediction band,
which can only make the distance look larger. The BBR row is the one
place that band is comparable to the published one, and the combined
(quadrature) metric printed under the table states what the distance
reduces to there. Two facts printed under the table carry the reading: the
BBR difference sits at a few hundredths of the published uncertainty,
small by construction since the registry's dynamic coefficients are
anchored to this group's own measurement, which is exactly why that row
carries the weaker class label; and Bothwell's two analysis methods
differ from each other by more than either differs from this tool's
single prediction, which lies between them. The uncovered rows follow
from the committed case's own isolation note, and `docs/roadmap.md`
frames when each becomes a shippable physics package.

In [7]:
sig_bbr_pub = (bbr_case.published_shift_hi - bbr_case.published_shift_lo) / 2.0
d_bbr = bbr_case.predicted_shift_nominal - bbr_case.published_shift_nominal

pred_mm20 = grav_case["predicted_slope_per_mm"] * 1e20
ma = grav_case["measured_slope_method_a"]
mb = grav_case["measured_slope_method_b"]
ma_n, ma_s = ma["nominal"] * 1e20, (ma["hi"] - ma["lo"]) / 2.0 * 1e20
mb_n, mb_s = mb["nominal"] * 1e20, (mb["hi"] - mb["lo"]) / 2.0 * 1e20

header = ("row", "published", "this tool", "delta (tool-pub)", "sigma", "verdict / class")
rows = [
    (
        "BBR, fractional (Aeppli Table I)",
        f"{bbr_case.published_shift_nominal:+.6e} +/- {sig_bbr_pub:.1e}",
        f"{bbr_case.predicted_shift_nominal:+.6e}",
        f"{d_bbr:+.1e}",
        f"{abs(d_bbr) / sig_bbr_pub:.2f}",
        f"{bbr_case.kpi_verdict}; {bbr_case.case_class}",
    ),
    (
        "redshift slope, method A (1e-20/mm)",
        f"{ma_n:+.1f} +/- {ma_s:.1f}",
        f"{pred_mm20:+.2f}",
        f"{pred_mm20 - ma_n:+.2f}",
        f"{grav_case['sigma_distance_method_a']:.2f}",
        f"{grav_case['kpi_verdict_method_a']}; {grav_case['case_class']}",
    ),
    (
        "redshift slope, method B (1e-20/mm)",
        f"{mb_n:+.1f} +/- {mb_s:.1f}",
        f"{pred_mm20:+.2f}",
        f"{pred_mm20 - mb_n:+.2f}",
        f"{grav_case['sigma_distance_method_b']:.2f}",
        f"{grav_case['kpi_verdict_method_b']}; {grav_case['case_class']}",
    ),
    (
        "DC-Stark gradient (Bothwell Table 1)",
        DC_STARK_ROW_VALUE,
        "awaiting a field characterization",
        "n/a",
        "n/a",
        "context row (no case object)",
    ),
]
widths = [max(len(r[i]) for r in [header, *rows]) for i in range(6)]
for r in [header, *rows]:
    print("  ".join(f"{cell:<{w}}" for cell, w in zip(r, widths, strict=True)))

ab_spread = ma_n - mb_n
ab_sigma = abs(ab_spread) / (ma_s**2 + mb_s**2) ** 0.5
print()
print(
    f"methods A and B differ from each other by {ab_spread:+.1f}e-20/mm "
    f"({ab_sigma:.2f} sigma combined); this tool's single prediction lies between them."
)
sig_bbr_tool = bbr_case.predicted_combined_uncertainty_fractional
bbr_sigma_combined = abs(d_bbr) / (sig_bbr_pub**2 + sig_bbr_tool**2) ** 0.5
print(
    f"the BBR difference is {abs(d_bbr) / sig_bbr_pub:.2f} sigma of the published "
    f"uncertainty alone, and {bbr_sigma_combined:.2f} sigma against the published "
    f"and prediction bands combined in quadrature (tool band +/- {sig_bbr_tool:.1e}); "
    "small by construction (the registry anchor traces to this group's own "
    "measurement), hence the weaker class label."
)
print()
print("uncovered rows, from the committed case's own isolation note:")
print(grav_case["isolation_note"])

row                                   published                  this tool                          delta (tool-pub)  sigma  verdict / class             
BBR, fractional (Aeppli Table I)      -4.841720e-15 +/- 7.3e-19  -4.841743e-15                      -2.3e-20          0.03   MET; arithmetic_reproduction
redshift slope, method A (1e-20/mm)   -9.8 +/- 2.3               -10.90                             -1.10             0.48   MET; reproducibility        
redshift slope, method B (1e-20/mm)   -12.8 +/- 2.7              -10.90                             +1.90             0.70   MET; reproducibility        
DC-Stark gradient (Bothwell Table 1)  +0.3(0.2)e-20/mm           awaiting a field characterization  n/a               n/a    context row (no case object)

methods A and B differ from each other by +3.0e-20/mm (0.85 sigma combined); this tool's single prediction lies between them.
the BBR difference is 0.03 sigma of the published uncertainty alone, and 0.02 sigma against the publish

## Where to go next

This slice held two computed rows and one context row from one real
platform, every number produced by the same engine calls a lab's own
`config.yaml` would trigger, and the classification stays exactly where
the record puts it: two reproducibility cases and zero blind
predictions across the project, with this notebook adding the composed,
single-platform view and no new class of claim.

- `notebooks/07_bbr_jila_arithmetic.ipynb`: the BBR row's full
  walkthrough, including the validity-window comparison against the
  open Lisdat dataset.
- `notebooks/09_bothwell_redshift.ipynb`: the redshift row's full
  walkthrough, geometry built by hand and both published measurements
  charted.
- `docs/validation.md`: the case-by-case validation record.
- `docs/roadmap.md`: the uncovered terms as future physics packages,
  and the blind-prediction partner ask.